In [1]:
import requests
import pandas as pd

documents = requests.get(
    'https://datatalks.club/faq/json/data-engineering-zoomcamp.json'
).json()

df = pd.DataFrame(documents)
df = df.rename(columns={'answer': 'text'})
df.head()

,id,course,section,question,text
0,9e508f2212,data-engineering-zoomcamp,General Course-Related Questions,Course: When does the course start?,A new cohort runs roughly January–April every ...
1,bfafa427b3,data-engineering-zoomcamp,General Course-Related Questions,Course: What are the prerequisites for this co...,"To get the most out of this course, you should..."
2,3f1424af17,data-engineering-zoomcamp,General Course-Related Questions,Course: Can I still join the course after the ...,"Yes, even if you don't register, you're still ..."
3,52217fc51b,data-engineering-zoomcamp,General Course-Related Questions,Course: I have registered for the Data Enginee...,You don't need a confirmation email. You're ac...
4,33fc260cd8,data-engineering-zoomcamp,General Course-Related Questions,Course: What can I do before the course starts?,Get the basic environment ready ahead of time:...


In [2]:
print(f"Total de documentos: {len(df)}")
print(f"\nSecciones únicas:")
print(df['section'].value_counts())

Total de documentos: 402

Secciones únicas:
section
Module 4: Analytics Engineering with dbt          68
Module 3: Data Warehousing                        48
Module 6: Spark                                   35
Module 1: Docker                                  33
Module 7: Streaming                               32
General Course-Related Questions                  26
Module 2: Workflow Orchestration                  26
Module 1: Postgres, pgAdmin & Python ingestion    25
Module 1: GCP setup & VM                          24
Environment & Setup                               23
Workshop 1 - dlthub                               18
Module 1: Terraform                               14
Project                                           12
Module 5: Data Platforms (Bruin)                  10
Module 1: Taxi Data (download & handling)          8
Name: count, dtype: int64


In [3]:
docs_example = [
    "Course starts on 15th Jan 2024",
    "Prerequisites listed on GitHub",
    "Submit homeworks after start date",
    "Registration not required for participation",
    "Setup Google Cloud and Python before course",
]

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english')
X = cv.fit_transform(docs_example)

names = cv.get_feature_names_out()

df_docs = pd.DataFrame(X.toarray(), columns=names).T
df_docs

,0,1,2,3,4
15th,1,0,0,0,0
2024,1,0,0,0,0
cloud,0,0,0,0,1
course,1,0,0,0,1
date,0,0,1,0,0
github,0,1,0,0,0
google,0,0,0,0,1
homeworks,0,0,1,0,0
jan,1,0,0,0,0
listed,0,1,0,0,0


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

cv = TfidfVectorizer(stop_words='english')
X = cv.fit_transform(docs_example)

names = cv.get_feature_names_out()

df_docs = pd.DataFrame(X.toarray(), columns=names).T
df_docs.round(2)

,0,1,2,3,4
15th,0.46,0.00,0.0,0.00,0.00
2024,0.46,0.00,0.0,0.00,0.00
cloud,0.00,0.00,0.0,0.00,0.46
course,0.37,0.00,0.0,0.00,0.37
date,0.00,0.00,0.5,0.00,0.00
github,0.00,0.58,0.0,0.00,0.00
google,0.00,0.00,0.0,0.00,0.46
homeworks,0.00,0.00,0.5,0.00,0.00
jan,0.46,0.00,0.0,0.00,0.00
listed,0.00,0.58,0.0,0.00,0.00


In [6]:
query = "Do I need to know python to sign up for the January course?"

q = cv.transform([query])
q.toarray()

array([[0.        , 0.        , 0.        , 0.62791376, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.77828292, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ]])

In [7]:
cv.vocabulary_

{'course': 3,
 'starts': 17,
 '15th': 0,
 'jan': 8,
 '2024': 1,
 'prerequisites': 11,
 'listed': 9,
 'github': 5,
 'submit': 18,
 'homeworks': 7,
 'start': 16,
 'date': 4,
 'registration': 13,
 'required': 14,
 'participation': 10,
 'setup': 15,
 'google': 6,
 'cloud': 2,
 'python': 12}

In [8]:
X.dot(q.T).toarray()

array([[0.23490553],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.59579005]])

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity(X, q)

array([[0.23490553],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.59579005]])

In [10]:
fields = ['section', 'question', 'text']
transformers = {}
matrices = {}

for field in fields:
    cv = TfidfVectorizer(stop_words='english', min_df=3)
    X = cv.fit_transform(df[field])

    transformers[field] = cv
    matrices[field] = X

In [11]:
for field in fields:
    print(f"{field}: {len(transformers[field].vocabulary_)} palabras")

section: 32 palabras
question: 252 palabras
text: 1352 palabras


In [12]:
print("Tamaño de las matrices generadas (Documentos, Palabras en el vocabulario):")
for field in fields:
    print(f"{field}: {matrices[field].shape}")

Tamaño de las matrices generadas (Documentos, Palabras en el vocabulario):
section: (402, 32)
question: (402, 252)
text: (402, 1352)


In [13]:
query = "I just signed up. Is it too late to join the course?"

q = transformers['text'].transform([query])
score = cosine_similarity(matrices['text'], q).flatten()

In [14]:
import numpy as np

mask = (df.course == 'data-engineering-zoomcamp').values
score = score * mask

In [15]:
idx = np.argsort(-score)[:10]
df.iloc[idx].text

13     No, as long as you complete the peer-reviewed ...
0      A new cohort runs roughly January–April every ...
16     No, late submissions are not allowed. However,...
307    The latest filename is just `taxi_zone_lookup....
22     You will have two attempts for a project.\n\n-...
24     There will be two announcements in the course ...
262    This error could result if you are using a `SE...
3      You don't need a confirmation email. You're ac...
99     In join queries, if you mention the column nam...
15     Deadlines are published per cohort. Find them ...
Name: text, dtype: str

In [16]:
query = "I just signed up. Is it too late to join the course?"

boost = {'question': 3.0}

score = np.zeros(len(df))

for f in fields:
    b = boost.get(f, 1.0)
    q = transformers[f].transform([query])
    s = cosine_similarity(matrices[f], q).flatten()
    score = score + b * s

In [17]:
idx = np.argsort(-score)[:5]
print(score[idx])

[3.5871588  3.54287519 3.5        3.5        3.18694851]


In [18]:
# 1. Aplicamos el filtro duro (Filtering) para DE Zoomcamp
mask = (df.course == 'data-engineering-zoomcamp').values
score = score * mask

# 2. Ordenamos descendentemente y tomamos los 5 índices más altos
idx = np.argsort(-score)[:5]

# 3. Imprimimos para ver el efecto del Boosting
print("Top 5 resultados con Boosting en 'question':\n")
for i, index in enumerate(idx):
    print(f"--- Rango {i+1} (Puntaje: {score[index]:.3f}) ---")
    print(f"Pregunta original: {df.iloc[index]['question']}")
    print(f"Texto: {df.iloc[index]['text'][:100]}...\n")

Top 5 resultados con Boosting en 'question':

--- Rango 1 (Puntaje: 3.587) ---
Pregunta original: Edit Course Profile.
Texto: The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to ...

--- Rango 2 (Puntaje: 3.543) ---
Pregunta original: Course: What are the prerequisites for this course?
Texto: To get the most out of this course, you should have:

- Basic coding experience
- Familiarity with S...

--- Rango 3 (Puntaje: 3.500) ---
Pregunta original: Course: What can I do before the course starts?
Texto: Get the basic environment ready ahead of time:

- Google Cloud account (free trial — see the GCP set...

--- Rango 4 (Puntaje: 3.500) ---
Pregunta original: How can we contribute to the course?
Texto: - [Star the repository](https://github.com/DataTalksClub/data-engineering-zoomcamp).
- Share it with...

--- Rango 5 (Puntaje: 3.187) ---
Pregunta original: Course: When does the course start?
Texto: A new cohort runs roughly January–April every

In [19]:
filters = {
    'course': 'data-engineering-zoomcamp',
}

for field, value in filters.items():
    mask = (df[field] == value).values
    score = score * mask

In [20]:
idx = np.argsort(-score)[:10]
results = df.iloc[idx]
results.to_dict(orient='records')

[{'id': '16005581f2',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Edit Course Profile.',
  'text': 'The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to be a nickname or your real name if you prefer. Your entry on the Leaderboard is the one highlighted in light green.\n\nThe Certificate name should be your actual name that you want to appear on your certificate after completing the course.\n\nThe "Display on Leaderboard" option indicates whether you want your name to be listed on the course leaderboard.'},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'text': 'To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering 

In [21]:
class TextSearch:

    def __init__(self, text_fields):
        self.text_fields = text_fields
        self.matrices = {}
        self.vectorizers = {}

    def fit(self, records, vectorizer_params={}):
        self.df = pd.DataFrame(records)
        if 'answer' in self.df.columns:
            self.df = self.df.rename(columns={'answer': 'text'})

        for f in self.text_fields:
            cv = TfidfVectorizer(**vectorizer_params)
            X = cv.fit_transform(self.df[f])
            self.matrices[f] = X
            self.vectorizers[f] = cv

    def search(self, query, n_results=10, boost={}, filters={}):
        score = np.zeros(len(self.df))

        for f in self.text_fields:
            b = boost.get(f, 1.0)
            q = self.vectorizers[f].transform([query])
            s = cosine_similarity(self.matrices[f], q).flatten()
            score = score + b * s

        for field, value in filters.items():
            mask = (self.df[field] == value).values
            score = score * mask

        idx = np.argsort(-score)[:n_results]
        results = self.df.iloc[idx]
        return results.to_dict(orient='records')

In [22]:
# 1. Instanciamos el motor indicándole los campos de búsqueda
index = TextSearch(
    text_fields=['section', 'question', 'text']
)

# 2. Entrenamos el motor (Asegúrate de pasarle los datos correctos)
# El workshop asume que 'documents' es la lista cruda de diccionarios que descargaste el primer día
index.fit(documents, vectorizer_params={'stop_words': 'english', 'min_df': 3})

# 3. Ejecutamos una búsqueda de producción
results = index.search(
    query='I just signed up. Is it too late to join the course?',
    n_results=5,
    boost={'question': 3.0},
    filters={'course': 'data-engineering-zoomcamp'}
)

print(results)

[{'id': '16005581f2', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Edit Course Profile.', 'text': 'The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to be a nickname or your real name if you prefer. Your entry on the Leaderboard is the one highlighted in light green.\n\nThe Certificate name should be your actual name that you want to appear on your certificate after completing the course.\n\nThe "Display on Leaderboard" option indicates whether you want your name to be listed on the course leaderboard.'}, {'id': 'bfafa427b3', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: What are the prerequisites for this course?', 'text': 'To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is nec

In [23]:
from sklearn.decomposition import TruncatedSVD

X = matrices['text']
cv = transformers['text']

svd = TruncatedSVD(n_components=16)
X_emb = svd.fit_transform(X)

X_emb[0]

array([ 0.17476997, -0.1454431 ,  0.16878697, -0.4289849 , -0.07863393,
       -0.09295805,  0.02901295,  0.04592845,  0.15318146,  0.17886146,
        0.14145385,  0.01127088,  0.12815024, -0.07738717, -0.02904116,
       -0.12411513])

In [24]:
query = 'I just signed up. Is it too late to join the course?'

Q = cv.transform([query])
Q_emb = svd.transform(Q)
Q_emb[0]

array([ 0.03316751, -0.024988  ,  0.02108269, -0.06445405, -0.01371522,
       -0.04018482,  0.01843668,  0.02665291,  0.02422118,  0.07533705,
        0.10441924, -0.02963546,  0.05348459, -0.06980574, -0.05398798,
       -0.08250531])

In [25]:
score = cosine_similarity(X_emb, Q_emb).flatten()
idx = np.argsort(-score)[:10]
list(df.loc[idx].text)

['The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to be a nickname or your real name if you prefer. Your entry on the Leaderboard is the one highlighted in light green.\n\nThe Certificate name should be your actual name that you want to appear on your certificate after completing the course.\n\nThe "Display on Leaderboard" option indicates whether you want your name to be listed on the course leaderboard.',
 "No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.",
 'No, as long as you complete the peer-reviewed capstone projects on time, you can receive the certificate. You do not need to do the homeworks if you join late, for example.',
 'Collaboration is not allowed for the capstone submission. However, you can 

In [26]:
from sklearn.decomposition import NMF

nmf = NMF(n_components=16)
X_emb = nmf.fit_transform(X)
X_emb[0]

array([0.        , 0.        , 0.        , 0.35742881, 0.        ,
       0.        , 0.00239224, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.00599009,
       0.        ])

In [27]:
Q = cv.transform([query])
Q_emb = nmf.transform(Q)
Q_emb[0]

score = cosine_similarity(X_emb, Q_emb).flatten()
idx = np.argsort(-score)[:10]
list(df.loc[idx].text)

['Each submitted project will be evaluated by three randomly assigned students who have also submitted the project.\n\nYou will also be responsible for grading projects from three fellow students yourself. Please note that not complying with this rule will result in failing to achieve the Certificate at the end of the course.\n\nThe final grade you receive will be the median score of the grades from peer reviewers.\n\nThe peer review criteria for evaluating or being evaluated must follow the guidelines defined [here](https://github.com/DataTalksClub/data-engineering-zoomcamp/tree/main/week_7_project#peer-review-criteria).',
 'Collaboration is not allowed for the capstone submission. However, you can discuss ideas and get feedback from peers in the forums or Slack channels.',
 'The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to be a nickname or your real name if you prefer. Your entry on the Leaderboard is the one highlighted in light gre

In [28]:
import torch
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\Sergio\Documents\Coding\learning\4. LLM_Zoomcamp\workshop-search-engine\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sergio\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     | Details
-------------------------------------------+------------+--------
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |        
cls.predictions.bias                       | UNEXPECTED |        
cls.seq_relationship.bias                  | UNEXPECTED |        
cls.predictions.transform.dense.bias       | UNEXPECTED |        
cls.seq_relationship.weight                | UNEXPECTED |        
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |        
cls.predictions.transform.dense.weight     | UNEXPECTED |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [30]:
texts = [
    "Yes, we will keep all the materials after the course finishes.",
    "You can follow the course at your own pace after it finishes",
]

encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
encoded_input

{'input_ids': tensor([[  101,  2748,  1010,  2057,  2097,  2562,  2035,  1996,  4475,  2044,
          1996,  2607, 12321,  1012,   102],
        [  101,  2017,  2064,  3582,  1996,  2607,  2012,  2115,  2219,  6393,
          2044,  2009, 12321,   102,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]])}

In [31]:
with torch.no_grad():
    outputs = model(**encoded_input)
    hidden_states = outputs.last_hidden_state

sentence_embeddings = hidden_states.mean(dim=1)
sentence_embeddings.shape

torch.Size([2, 768])

In [32]:
def make_batches(seq, n):
    result = []
    for i in range(0, len(seq), n):
        batch = seq[i:i+n]
        result.append(batch)
    return result

In [33]:
from tqdm.auto import tqdm

texts = df['text'].tolist()
text_batches = make_batches(texts, 4)

all_embeddings = []

for batch in tqdm(text_batches):
    encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        outputs = model(**encoded_input)
        hidden_states = outputs.last_hidden_state
        batch_embeddings = hidden_states.mean(dim=1)
        batch_embeddings_np = batch_embeddings.cpu().numpy()
        all_embeddings.append(batch_embeddings_np)

X_emb = np.vstack(all_embeddings)
X_emb.shape

  0%|          | 0/101 [00:00<?, ?it/s]

(402, 768)

In [34]:
query = "I just signed up. Is it too late to join the course?"

encoded_q = tokenizer([query], padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    outputs = model(**encoded_q)
    Q_emb = outputs.last_hidden_state.mean(dim=1).numpy()

score = cosine_similarity(X_emb, Q_emb).flatten()
idx = np.argsort(-score)[:10]
list(df.loc[idx].text)

['There is only ONE project for this Zoomcamp. You do not need to submit or create two projects.\n\nThere are simply TWO chances to pass the course. You can use the Second Attempt if you:\n\n- Fail the first attempt\n- Do not have the time due to other engagements such as holidays or sickness to enter your project into the first attempt.\n\n**Project Evaluation - Reproducibility**\n\nEven with thorough documentation, ensuring that a peer reviewer can follow your steps can be challenging. Here’s how this criterion will be evaluated:\n\n> "Ideally yes, you should try to re-run everything. But I understand that not everyone has time to do it, so if you check the code by looking at it and try to spot errors, places with missing instructions, and so on - then it\'s already great."\n\n**Certificates: How do I get it?**\n\nSee the `certificate.mdx` file.',
 'The display name listed on the leaderboard is an auto-generated randomized name. You can edit it to be a nickname or your real name if y

In [35]:
# Embeddings de la columna question
q_texts = df['question'].tolist()
q_batches = make_batches(q_texts, 4)
q_embeddings = []

for batch in tqdm(q_batches):
    enc = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        out = model(**enc)
        emb = out.last_hidden_state.mean(dim=1).numpy()
        q_embeddings.append(emb)

Q_emb_docs = np.vstack(q_embeddings)

  0%|          | 0/101 [00:00<?, ?it/s]

In [36]:
query = "I just signed up. Is it too late to join the course?"

encoded_q = tokenizer([query], padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    outputs = model(**encoded_q)
    query_emb = outputs.last_hidden_state.mean(dim=1).numpy()

score = cosine_similarity(Q_emb_docs, query_emb).flatten()

# Filtro por curso
mask = (df.course == 'data-engineering-zoomcamp').values
score = score * mask

idx = np.argsort(-score)[:5]
df.iloc[idx][['question', 'text']]

,question,text
3,Course: I have registered for the Data Enginee...,You don't need a confirmation email. You're ac...
14,Certificate - Can I follow the course in a sel...,"No, you can only get a certificate if you fini..."
48,My country isn't supported by GCP / my card is...,"GCP isn't available in some countries, and som..."
8,Course: Can I get support if I take the course...,"Yes, the Slack channel remains open and you ca..."
6,Course: Is the current cohort going to be diff...,Most of the syllabus stays consistent year to ...


In [37]:
# Comparamos la pregunta del usuario contra las preguntas de la BD
score = cosine_similarity(Q_emb_docs, Q_emb).flatten()
idx = np.argsort(-score)[:10]

# Imprimimos los documentos resultantes (puedes imprimir el text o el question para verificar)
list(df.loc[idx].question)

['Course: I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 "My country isn't supported by GCP / my card is rejected. What can I do?",
 'Course: Can I get support if I take the course in the self-paced mode?',
 'Course: Is the current cohort going to be different from the previous cohort?',
 'Certificate: Do I need to do the homeworks to get the certificate?',
 'Course: What are the prerequisites for this course?',
 'How is my capstone project going to be evaluated?',
 'Course: Can I still join the course after the start date?',
 'Course: How many hours per week am I expected to spend on this course?']